# Model Training and Execution Testbed as preparation for Ensembling

In [ ]:
import os
import sys
sys.path.insert(0, '../../')

import time
import shutil

import numpy as np
import pandas as pd
import torch
import tqdm.notebook as tqdm

import CRE_utils

from boda2 import boda


# Prepare Custom Dataset

In [ ]:
file_path = "../_data_evaluation/CRE/41586_2024_8070_MOESM4_ESM.txt"
output_folder = "_data/derived_datasets/"

os.makedirs(output_folder, exist_ok=True)

In [ ]:
def scoring_function(elem):
    if "K562_log2FC" in elem.index:
        K562 = elem["K562_log2FC"]
        HepG2 = elem["HepG2_log2FC"]
        SKNSH = elem["SKNSH_log2FC"]
    else:
        K562 = elem["K562_preds"]
        HepG2 = elem["HepG2_preds"]
        SKNSH = elem["SKNSH_preds"]

    score = K562
    return score

data = pd.read_csv(file_path, delimiter="\t")
data = data.loc[ data.loc[:, ['K562_lfcSE', 'HepG2_lfcSE', 'SKNSH_lfcSE']].max(axis=1) < 1.0 ]
data = data.loc[ data['sequence'].str.len() == 200 ].reset_index(drop=True)
data["score"] = data.apply(scoring_function, axis=1)

data = data.loc[np.logical_not(np.isnan(data["score"]))]

print(f"Num sequences is {len(data)}")

# Selection Tests


## Model Selection

In [ ]:
n_init = 34
n_rounds = 3
q = 16

In [ ]:
def model_based_optimization_loop(data, n_init, n_rounds, q, identifier="SingleRun", model_id=0, output_directory=None):
    split = [0.75, 0.25, 0.0]

    time_stamp = CRE_utils.get_time_stamp()

    output_file = None
    if output_directory:        
        # output_dir = f"_output_backup/"
        output_dir = output_directory
        output_file = os.path.join(output_dir, f"{identifier}_{time_stamp}.tsv")
        os.makedirs(output_dir, exist_ok=True)    

    print(f"{identifier}: Preparing Sequences")
    sequences = data["sequence"]
    seq_tensor  = torch.stack([boda.common.utils.dna2tensor(seq) for seq in tqdm.tqdm(sequences, total=len(sequences)) ], dim=0)
    seq_dataset = torch.utils.data.TensorDataset(seq_tensor)
    seq_loader  = torch.utils.data.DataLoader(seq_dataset, batch_size=128)
    
    print(f"{identifier}: Creating Initial Dataset")
    initial_data_indexes = np.random.choice(np.arange(len(data)), replace=False, size=n_init)
    
    initial_data = data.iloc[initial_data_indexes]
    
    result_data = pd.DataFrame(initial_data)
    # result_data_indexes = initial_data_indexes.copy()    

    result_data["round"] = 0
    
    for iR in range(n_rounds):
        train_indexes, validation_indexes, test_indexes = CRE_utils.split_data(result_data, split=split)
        train_data = pd.DataFrame(result_data.iloc[train_indexes])
        val_data = pd.DataFrame(result_data.iloc[validation_indexes])
        test_data = pd.DataFrame(result_data.iloc[test_indexes])
        if split[2] == 0.0:
            test_data = pd.DataFrame(result_data.iloc[:2])
        
        candidate_sequences = data.loc[np.logical_not(data["sequence"].isin(list(result_data["sequence"])))]

        output_folder = os.path.join("../_data_evaluation/CRE/custom/training_data/", f"{time_stamp}_{identifier}")
        os.makedirs(output_folder, exist_ok=True)

        model_path = CRE_utils.train_model(train_data, val_data, test_data, output_folder, min_epochs=60, max_epochs=200, identifier=identifier, model_id=model_id)
        shutil.rmtree(output_folder, ignore_errors=True)
        
        model, flank_builder = CRE_utils.prepare_model(model_path, model_dir=os.path.join("./_intermediate/", identifier), model_id=model_id)

        pred_df =  CRE_utils.evaluate_model(sequences=sequences, sequence_loader=seq_loader, model=model, flank_builder=flank_builder, model_id=model_id)
        # print(f"{identifier} {iR}: Evaluated Model")
        
        
        pred_df["score"] = pred_df.apply(scoring_function, axis=1)
        pred_df = pred_df.sort_values(by="score", ascending=False)
    
        candidates = pred_df.iloc[:q]
        
        new_data = pd.DataFrame(data.loc[candidates.index])
        new_data["round"] = iR + 1
        result_data = pd.concat((result_data, new_data))        
        
        if output_file:
            result_data.to_csv(output_file, sep="\t")
        print(f"{identifier} {iR}: Currently highest score {np.max(result_data['score'])}")
    
        
    return result_data


In [ ]:
identifier = "Model"

result_data_model = []
n_runs = 50

time_stamp = CRE_utils.get_time_stamp()
output_directory = f"_output/{identifier}_{n_runs}_runs_{time_stamp}/"

start = time.time()
for iX in range(n_runs):
    cur_result_data = model_based_optimization_loop(data, n_init, n_rounds, q, f"model_{iX:03d}", iX + 1, output_directory)
    result_data_model.append(cur_result_data)    

end = time.time()
# result_data_model = results 

print(f"Evaluation completed: {n_runs} took {end - start} s")
with open("output.txt", "w") as file:
    file.write(f"Evaluation completed: {n_runs} took {end - start} s")

In [ ]:

from multiprocessing.pool import ThreadPool

n_runs = 20

identifier = "Model"
time_stamp = CRE_utils.get_time_stamp()
output_directory = f"_output/{identifier}_{n_runs}_runs_{time_stamp}/"


pool_size = np.min([torch.cuda.device_count() * 2, os.cpu_count() - 6])

start = time.time()

# with Pool(pool_size) as pool:
with ThreadPool(pool_size) as pool:
    
    argument_list = [(data, n_init, n_rounds, q, f"model_{iX:03d}", iX, output_directory) for iX in range(n_runs)]
    # pool.apply(model_based_optimization_loop, argument_list[0])
    results = pool.starmap(model_based_optimization_loop, argument_list)   

end = time.time()
result_data_model = results 

print(f"Evaluation completed: {n_runs} took {end - start} s")

with open("output_2.txt", "w") as file:
    file.write(f"Evaluation completed: {n_runs} took {end - start} s\n{CRE_utils.get_time_stamp()}")

In [ ]:
CRE_utils.save_results(result_data_model, identifier="Model_20_runs_fixed_initial_indice")

# Random Selection Test

In [ ]:
n_init = 34
n_rounds = 3
q = 16

In [ ]:
def random_optimization_loop(data, n_init, n_rounds, q):

    initial_samples_indexes = np.random.choice(np.arange(len(data)), replace=False, size=n_init)
    initial_samples = data.iloc[initial_samples_indexes]
    
    result_data = pd.DataFrame(initial_samples)
    result_data_indexes = initial_samples_indexes.copy()
    result_data["round"] = 0
    
    for iR in range(n_rounds):
    
        batch_samples_indexes = []
        iQ = 0
        while len(batch_samples_indexes) < q:
            proposals = np.random.choice(np.arange(len(data)), replace=False, size=q - len(batch_samples_indexes))
            mask = np.array(list(map(lambda elem: elem not in result_data_indexes and elem not in batch_samples_indexes, proposals)))
            batch_samples_indexes.extend(proposals[mask])
            print(f"Creating batch in round {iR} ({iQ}) {len(batch_samples_indexes)} samples")
            iQ += 1
    
        new_data = data.iloc[batch_samples_indexes]
        new_data = pd.DataFrame(new_data)
        new_data["round"] = iR + 1
        print(f"Length before {len(result_data)}")
        result_data = pd.concat((result_data, new_data), axis=0)
        print(f"Length after {len(result_data)}")
        result_data_indexes = np.concatenate((result_data_indexes, batch_samples_indexes))
            
        print(f"Completed round {iR}")
        
    return result_data


In [ ]:
result_data_random = []
for iX in range(100):
    cur_result_data = random_optimization_loop(data, n_init, n_rounds, q)
    result_data_random.append(cur_result_data)

In [ ]:
CRE_utils.save_results(result_data_random, identifier=f"Random_{n_rounds}_rounds")